In [2]:
import brainpy as bp
import brainpy.math as bm
import numpy as np


In [6]:
class LIFNeuron(bp.dyn.NeuDyn):
  def __init__(self, size, V_rest=0., V_reset=-5., V_th=20., R=1., tau=10., t_ref=5., **kwargs):
    super(LIFNeuron, self).__init__(size=size, **kwargs)

    # initialize parameters
    self.V_rest = V_rest
    self.V_reset = V_reset
    self.V_th = V_th
    self.R = R
    self.tau = tau
    self.t_ref = t_ref

    # initialize variables
    self.V = bm.Variable(bm.random.randn(self.num) + V_reset)
    self.t_last_spike = bm.Variable(bm.ones(self.num) * -1e7)
    self.refractory = bm.Variable(bm.zeros(self.num, dtype=bool))
    self.spike = bm.Variable(bm.zeros(self.num, dtype=bool))

    # integral function
    self.integral = bp.odeint(f=self.derivative, method='exp_auto')

  def derivative(self, V, t, Iext):
    dvdt = (-V + self.V_rest + self.R * Iext) / self.tau
    return dvdt

  def update(self, x=None): # x is external input, I_ext. Units of pA
    _t = bp.share['t']
    _dt = bp.share['dt']
    x = 0. if x is None else x

    # Whether the neurons are in the refractory period
    refractory = (_t - self.t_last_spike) <= self.t_ref

    # compute the membrane potential
    V = self.integral(self.V, _t, x, dt=_dt)

    # computed membrane potential is valid only when the neuron is not in the refractory period
    V = bm.where(refractory, self.V, V)

    # update the spiking state
    spike = self.V_th <= V
    self.spike.value = spike

    # update the last spiking time
    self.t_last_spike.value = bm.where(spike, _t, self.t_last_spike)

    # update the membrane potential and reset spiked neurons
    self.V.value = bm.where(spike, self.V_reset, V)

    # update the refractory state
    self.refractory.value = bm.logical_or(refractory, spike)

In [7]:
group = Neuron(10)

runner = bp.DSRunner(group, monitors=['V'])

inputs = np.ones(int(200. / bm.get_dt())) * 22.  # 200 ms
runner.run(inputs=inputs)

bp.visualize.line_plot(runner.mon.ts, runner.mon.V, show=True)

NameError: name 'Neuron' is not defined

In [22]:
class DistanceDependentSynapse(bp.dyn.SynConn):
 ........................


In [23]:
class NetworkModel(bp.DynSysGroup):
    ....................

In [24]:

................run..............


/home/brendan/OneDrive/Masters/Code/Vortices/Julia/WRCircuit/.CondaPkg/env/lib/python3.11/site-packages/brainpy/_src/connect/random_conn.py:514: DeprecationWarning: invalid escape sequence '\_'
  """Build a Watts–Strogatz small-world graph.
/home/brendan/OneDrive/Masters/Code/Vortices/Julia/WRCircuit/.CondaPkg/env/lib/python3.11/site-packages/brainpy/_src/connect/random_conn.py:658: DeprecationWarning: invalid escape sequence '\_'
  """Build a random graph according to the Barabási–Albert preferential


TypeError: TwoEndConnector.__init__() got an unexpected keyword argument 'pre_size'